In [4]:
from pathlib import Path
import pandas as pd

PLAN_BASE = Path(
    "/home/SpeakerRec/BioVoice/data/datasets/ASVspoof5_tars/ASVspoof5_protocols/train_dev_16_systems_outputs"
)
OUT_TXT = Path.cwd() / "bonafide_test_distribution_check.txt"

thresholds = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
lines = []

for part in ["train", "dev"]:
    path = PLAN_BASE / part / "selected_utterances_plan.csv"
    df = pd.read_csv(path)

    lines.append(f"===== {part.upper()} =====")
    lines.append(f"manifest_path: {path}")
    lines.append("")

    test_df = df[df["split"] == "test"].copy()

    lines.append("target_class counts on test:")
    lines.append(test_df["target_class"].value_counts().sort_index().to_string())
    lines.append("")

    if "label" in df.columns:
        lines.append("label counts on test:")
        lines.append(test_df["label"].value_counts().sort_index().to_string())
        lines.append("")

    bona = test_df[test_df["target_class"] == "bonafide"].copy()
    lines.append(f"bonafide test rows: {len(bona)}")

    if len(bona) > 0:
        counts = bona.groupby("speaker_id").size().sort_values(ascending=False)
        lines.append(f"bonafide test speakers: {len(counts)}")
        lines.append(f"bonafide max utts per speaker: {counts.max()}")
        lines.append("bonafide top speakers:")
        lines.append(counts.head(20).to_string())
        lines.append("")

        lines.append("eligible bonafide speakers by threshold:")
        for t in thresholds:
            n = int((counts >= t).sum())
            lines.append(f"  threshold >= {t:2d}: {n} speakers")

        best_threshold = int(counts.max())
        best_speakers = counts[counts == best_threshold]
        lines.append("")
        lines.append(f"largest feasible bonafide threshold: {best_threshold}")
        lines.append("speakers at largest feasible threshold:")
        lines.append(best_speakers.to_string())
    else:
        lines.append("No bonafide rows found on test with target_class == 'bonafide'")

    lines.append("")
    lines.append("=" * 80)
    lines.append("")

OUT_TXT.write_text("\n".join(lines), encoding="utf-8")
print(f"Saved report to: {OUT_TXT}")

Saved report to: /home/SpeakerRec/BioVoice/redimnet/deepfakes/bonafide_test_distribution_check.txt
